In [15]:
import pandas as pd
import os
from sqlalchemy import create_engine, text  # Fixed: Added text import
import logging
import time

for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    filename="Desktop/data/logs/ingestion_db.log",
    level=logging.DEBUG,
    format="%(asctime)s - %(levelname)s - %(message)s",
    filemode="a"
)

logger = logging.getLogger("data_ingestion")
logging.getLogger("IPython").setLevel(logging.WARNING)

engine = create_engine('sqlite:///inventory.db')

def ingest_db(df, table_name, engine):
    df.to_sql(table_name, con=engine, if_exists='append', index=False, chunksize=50000)

data_directory = 'Desktop/data/'

def load_raw_data():
    """This function loads CSV as dataframe & ingests into db"""
    start = time.time()
    for file in os.listdir(data_directory):
        if file.endswith('.csv'):
            file_name_clear = file[:-4]
            
            with engine.connect() as conn:
                conn.execute(text(f"DROP TABLE IF EXISTS {file_name_clear};"))
                conn.commit()
                
            print(f"Starting ingestion for: {file}")
            
            for chunk in pd.read_csv(data_directory + file, chunksize=100000):
                logger.info('Ingesting file in db')  # Fixed: Changed logging to logger
                ingest_db(chunk, file_name_clear, engine)
                
    end = time.time()
    total_time = (end - start) / 60
    logger.info('----------------Ingestion Complete----------------')  # Fixed: Changed to logger
    logger.info(f'\nTotal Time Taken: {total_time:.2f} minutes')        # Fixed: Added .2f for clean formatting
    print(f"✅ All files processed completely in {total_time:.2f} minutes!")

if __name__ == '__main__':
    load_raw_data()

Starting ingestion for: begin_inventory.csv
Starting ingestion for: end_inventory.csv
Starting ingestion for: purchases.csv
Starting ingestion for: purchase_prices.csv
Starting ingestion for: sales.csv
Starting ingestion for: vendor_invoice.csv
✅ All files processed completely in 3.78 minutes!


In [7]:
engine = create_engine('sqlite:///inventory.db')

In [3]:
# Use the exact same path for both listing and reading
data_directory = 'Desktop/data/'

for file in os.listdir(data_directory):
    if file.endswith('.csv'):
        # Combine the full directory path with the file name
        df = pd.read_csv(data_directory + file)
        print(f"{file}: {df.shape}")

begin_inventory.csv: (206529, 9)
end_inventory.csv: (224489, 9)
purchases.csv: (2372474, 16)
purchase_prices.csv: (12261, 9)
sales.csv: (12825363, 14)
vendor_invoice.csv: (5543, 10)


In [4]:
import os
import pandas as pd
# 1. Import the text wrapper
from sqlalchemy import text 

def ingest_db(df, table_name, engine):
    df.to_sql(table_name, con=engine, if_exists='append', index=False, chunksize=50000)

data_directory = 'Desktop/data/'

for file in os.listdir(data_directory):
    if file.endswith('.csv'):
        file_name_clear = file[:-4]
        
        # 2. Wrap the SQL string in text() right here
        with engine.connect() as conn:
            conn.execute(text(f"DROP TABLE IF EXISTS {file_name_clear};"))
            conn.commit() # Good practice: commit changes in SQLAlchemy 2.0
        
        print(f"Starting ingestion for: {file}")
        
        for chunk in pd.read_csv(data_directory + file, chunksize=100000):
            ingest_db(chunk, file_name_clear, engine)
            
        print(f"✅ Finished ingesting: {file_name_clear}\n")

Starting ingestion for: begin_inventory.csv
✅ Finished ingesting: begin_inventory

Starting ingestion for: end_inventory.csv
✅ Finished ingesting: end_inventory

Starting ingestion for: purchases.csv
✅ Finished ingesting: purchases

Starting ingestion for: purchase_prices.csv
✅ Finished ingesting: purchase_prices

Starting ingestion for: sales.csv
✅ Finished ingesting: sales

Starting ingestion for: vendor_invoice.csv
✅ Finished ingesting: vendor_invoice



In [13]:
data_directory = 'Desktop/data/'

def load_raw_data():
    '''this function's gonna load CSV as dataframe & ingest into db'''
    start = time.time()
    for file in os.listdir(data_directory):
        if file.endswith('.csv'):
            file_name_clear = file[:-4]
            
            # 2. Wrap the SQL string in text() right here
            with engine.connect() as conn:
                conn.execute(text(f"DROP TABLE IF EXISTS {file_name_clear};"))
                conn.commit() # Good practice: commit changes in SQLAlchemy 2.0
            
            print(f"Starting ingestion for: {file}")
            
            for chunk in pd.read_csv(data_directory + file, chunksize=100000):
                logging.info(f'Ingesting {file} in db')
                ingest_db(chunk, file_name_clear, engine)
    end = time.time()
    total_time = (end-start)/60
    logging.info('--------------Ingestion Complete------------------')
    logging.info(f'\nTotal Time Taken: {total_time} minutes')
                
if __name__ == '__main__':
    load_raw_data()

Starting ingestion for: begin_inventory.csv
Starting ingestion for: end_inventory.csv
Starting ingestion for: purchases.csv
Starting ingestion for: purchase_prices.csv
Starting ingestion for: sales.csv
Starting ingestion for: vendor_invoice.csv
Finished ingesting: vendor_invoice

